# 07 - Interpretabilidade

**Objetivo:** gerar explicacoes globais com SHAP e locais com LIME para um modelo compativel.

As explicacoes indicam importancia para o modelo, nao causalidade medica.

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tcc_ecg.config import load_config
from tcc_ecg.features import get_feature_columns
from tcc_ecg.interpretability import run_lime_examples, run_shap_analysis
from tcc_ecg.models import split_by_folds
from tcc_ecg.paths import resolve_project_path

config = load_config()
tables_dir = resolve_project_path(config['outputs']['tables_dir'], config['project_root'])
models_dir = resolve_project_path(config['outputs']['models_dir'], config['project_root'])
frequency = int(config['data']['signal_frequency'])
features_path = resolve_project_path(config['outputs']['processed_dir'], config['project_root']) / f'features_{frequency}hz.parquet'

features = pd.read_parquet(features_path)
metrics = pd.read_csv(tables_dir / 'model_metrics.csv')
tree_metrics = metrics[metrics['model'].str.contains('random_forest|lightgbm|catboost') & metrics['split'].eq('test')]
if tree_metrics.empty:
    raise ValueError('Nenhum modelo de arvore encontrado para SHAP.')

model_name = tree_metrics.sort_values('f1_macro', ascending=False).iloc[0]['model']
pipeline = joblib.load(models_dir / f'{model_name}.joblib')
print('Modelo escolhido para interpretabilidade:', model_name)

Modelo escolhido para interpretabilidade: lightgbm_without_smote


In [2]:
feature_columns = get_feature_columns(features)
splits = split_by_folds(features, config)
X_train = splits['train'][feature_columns]
X_test = splits['test'][feature_columns]
y_test = splits['test']['target_id'].astype(int).to_numpy()
y_pred = pipeline.predict(X_test)

top_features = run_shap_analysis(pipeline, X_train, config, max_samples=300)
display(top_features.head(20))

run_lime_examples(
    pipeline=pipeline,
    X_train=X_train,
    X_examples=X_test,
    y_true=y_test,
    y_pred=y_pred,
    class_names=config['labels']['superclasses'],
    config=config,
)

C:\Users\victo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,feature,mean_abs_shap
144,age_clean,0.230948
40,aVR_median,0.140511
146,sex,0.111992
46,aVR_skew,0.100390
47,aVR_kurtosis,0.096968
112,V4_median,0.078820
123,V5_max,0.075526
42,aVR_p75,0.075264
88,V2_median,0.074843
100,V3_median,0.074278


C:\Users\victo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\victo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
